In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

In [ ]:
# Make sure to run this cell only once, which is at the start of the sesssion.
# !pip install tensorflow_model_optimization
# !pip install keras_tuner

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler, LabelBinarizer
from sklearn.manifold import TSNE
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, optimizers

# Install the missing package before importing
!pip install tensorflow_model_optimization
import tensorflow_model_optimization as tfmot
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 5.0 MB/s eta 0:00:00


In [ ]:
#Upload the zip file via 'Choose File' button ONLY ONCE PER SESSION. The uploaded zip file's name should be 'UCI HAR Dataset'

from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile
import os

for f in os.listdir("."):
    if f.endswith(".zip"):
        print("Extracting:", f)
        try:
            with zipfile.ZipFile(f, 'r') as z:
                z.extractall(".")
        except Exception as e:
            print("Error extracting", f, "->", str(e))

print("Done extracting.")
print("Directory now contains:")

print(os.listdir("."))

In [ ]:
print("All extracted items in current directory:")
print(os.listdir("."))

In [ ]:
# Run this cell only at the start of the session (ONLY ONCE AND LATER COMMENT IT OUT)
if os.path.exists("UCI-HAR Dataset"):
    os.rename("UCI-HAR Dataset", "UCI_HAR")

print("Renamed successfully.")
print(os.listdir("."))

In [ ]:
print(os.listdir("UCI_HAR"))
print(os.listdir("UCI_HAR/train")[:5])
print(os.listdir("UCI_HAR/train/Inertial Signals")[:5])

In [ ]:
DATA_DIR = "UCI_HAR"

def load_signals(split="train"):
    base = os.path.join(DATA_DIR, split, "Inertial Signals")
    files = sorted([os.path.join(base, f) for f in os.listdir(base) if f.endswith(".txt")])
    X = [np.loadtxt(f) for f in files]
    X = np.stack(X, axis=-1)
    return X

def load_y(split="train"):
    y_path = os.path.join(DATA_DIR, split, "y_"+split+".txt")
    return np.loadtxt(y_path).astype(int) - 1


In [ ]:
X_train = load_signals("train")
y_train = load_y("train")
X_test = load_signals("test")
y_test = load_y("test")

In [ ]:
print("Shapes:", X_train.shape, y_train.shape, X_test.shape, y_test.shape)


In [ ]:
n_channels = X_train.shape[-1]

plt.figure(figsize=(12, 6))
for i in range(n_channels):
    plt.plot(X_train[0,:,i] + i*5, label=f"ch{i}")
plt.title("Example sample (offset for visibility)")
plt.xlabel("Timestep (0..127)")
plt.legend(ncol=3)
plt.show()

In [ ]:
def build_cnn_lstm(input_shape=(128,9), num_classes=6, conv_filters=[64,128], lstm_units=128, dropout=0.3):
    inp = layers.Input(shape=input_shape,)
    x = inp

    for f in conv_filters:
        x = layers.Conv1D(filters=f, kernel_size=3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling1D(pool_size=2)(x)

    last_conv = x

    x = layers.Bidirectional(layers.LSTM(lstm_units, return_sequences=False))(x)
    x = layers.Dropout(dropout)(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(dropout)(x)

    out = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inp, out)

    model.last_conv_layer_name = last_conv.name.split("/")[0]

    return model

num_classes = len(np.unique(y_train))
model = build_cnn_lstm(input_shape=X_train.shape[1:], num_classes=num_classes)
model.summary()


In [ ]:
tf.keras.backend.clear_session()

model = build_cnn_lstm(input_shape=X_train.shape[1:], num_classes=num_classes)

print("Last conv layer stored for Grad-CAM:", model.last_conv_layer_name)

print("\nModel layers:")
for layer in model.layers:
    print(layer.name)

In [ ]:
CLASS_NAMES = ["Walking", "Walking Up", "Walking Down", "Sitting", "Standing", "Laying"]

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ReduceLROnPlateau(patience=3, factor=0.5, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=64,
    callbacks=callbacks
)

# Verify
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {acc:.2%}")

In [ ]:
def plot_history(h):
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(h.history["loss"], label="train_loss")
    plt.plot(h.history["val_loss"], label="val_loss")
    plt.legend(); plt.title("Loss")
    plt.subplot(1,2,2)
    plt.plot(h.history["accuracy"], label="train_acc")
    plt.plot(h.history["val_accuracy"], label="val_acc")
    plt.legend(); plt.title("Accuracy")

plot_history(history)

# FIX: use integer labels, not one-hot
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("Test acc:", test_acc)

y_pred = np.argmax(model.predict(X_test), axis=1)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sn.heatmap(cm, annot=True, fmt="d", cmap="Blues",
           xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

In [ ]:
probs = []

for i in range(10):
    sample = X_test[i].reshape(1,128,9)
    pred = model.predict(sample, verbose=0)
    probs.append(np.max(pred))

plt.plot(probs, marker='o')
plt.title("Prediction Confidence over Time")
plt.xlabel("Sample")
plt.ylabel("Confidence")
plt.show()

In [ ]:
import shap

NUM_CLASSES = 6
WINDOW_SIZE = 128
NUM_FEATURES = 9
CLASS_NAMES = ["Walking", "Walking Up", "Walking Down", "Sitting", "Standing", "Laying"]

def model_predict_flat(X_flat):
    X_reshaped = X_flat.reshape(-1, WINDOW_SIZE, NUM_FEATURES).astype(np.float32)
    preds = model.predict(X_reshaped, verbose=0)
    return preds  # shape (n, 6)

background_size = 50
background_flat = X_test[:background_size].reshape(background_size, -1).astype(np.float32)
explainer = shap.KernelExplainer(model_predict_flat, background_flat)

sample_idx = 34
sample_flat = X_test[sample_idx].reshape(1, -1).astype(np.float32)
shap_values = explainer.shap_values(sample_flat, nsamples=200)


print("Type:", type(shap_values))
print("Shape check:", np.array(shap_values).shape)

if isinstance(shap_values, list):
    sv_per_class = np.array(shap_values)[:, 0, :]  #
else:
    sv_per_class = shap_values.T

print("sv_per_class shape:", sv_per_class.shape)

if sv_per_class.shape != (6, 1152):
    print("Falling back to per-class manual SHAP extraction...")
    sv_per_class = np.zeros((NUM_CLASSES, WINDOW_SIZE * NUM_FEATURES))
    for c in range(NUM_CLASSES):
        def predict_class_c(X_flat, cls=c):
            X_r = X_flat.reshape(-1, WINDOW_SIZE, NUM_FEATURES).astype(np.float32)
            return model.predict(X_r, verbose=0)[:, cls]

        exp_c = shap.KernelExplainer(predict_class_c, background_flat)
        sv_c = exp_c.shap_values(sample_flat, nsamples=200)  # (1, 1152)
        sv_per_class[c] = sv_c[0]
        print(f"  Class {c} done, max val: {np.abs(sv_c).max():.5f}")

print("Final sv_per_class shape:", sv_per_class.shape)
print("Per-class sums:", sv_per_class.sum(axis=1))

class_relevance_signed = sv_per_class.sum(axis=1)
plt.figure(figsize=(7, 3))
colors = ["steelblue" if v >= 0 else "tomato" for v in class_relevance_signed]
plt.bar(CLASS_NAMES, class_relevance_signed, color=colors)
plt.axhline(0, color="k", linewidth=0.8)
plt.xlabel("Class")
plt.ylabel("SHAP net contribution")
plt.title("Class-level SHAP (signed) — this instance")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

class_relevance_magnitude = np.abs(sv_per_class).sum(axis=1)
plt.figure(figsize=(7, 3))
plt.bar(CLASS_NAMES, class_relevance_magnitude, color="salmon")
plt.xlabel("Class")
plt.ylabel("Sum |SHAP| (magnitude)")
plt.title("Class-level SHAP (magnitude)")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
correct_per_class = {}
preds_all = model.predict(X_test, verbose=0)
pred_labels = np.argmax(preds_all, axis=1)

for i, (true, pred) in enumerate(zip(y_test, pred_labels)):
    if true == pred and true not in correct_per_class:
        correct_per_class[true] = i
    if len(correct_per_class) == NUM_CLASSES:
        break

print("Best sample per class (correctly predicted):")
for cls, idx in correct_per_class.items():
    print(f"  {CLASS_NAMES[cls]}: idx={idx}")

In [ ]:
T, C = 128, 9
assert sv_per_class.shape == (6, T*C), f"Expected (6, {T*C}), got {sv_per_class.shape}"

predicted_class = int(np.argmax(model.predict(X_test[sample_idx:sample_idx+1], verbose=0)))
cls = predicted_class

sv_feat = sv_per_class[cls].reshape(T, C)

plt.figure(figsize=(14, 4))
sn.heatmap(
    sv_feat.T,
    center=0,
    cmap="RdBu_r",
    cbar_kws={'label': 'SHAP value'},
    xticklabels=list(range(0, T, 3)),
    yticklabels=["acc_x","acc_y","acc_z","gyro_x","gyro_y","gyro_z","total_acc_x","total_acc_y","total_acc_z"]
)
plt.xlabel("Timestep (0..127)")
plt.ylabel("Sensor Channel")
plt.title(f"SHAP Feature Heatmap — Predicted class: {CLASS_NAMES[cls]}")
plt.tight_layout()
plt.show()

importance_per_timestep = np.abs(sv_feat).sum(axis=1)
top_timesteps = np.argsort(importance_per_timestep)[::-1][:10]
print(f"Top 10 most important timesteps for {CLASS_NAMES[cls]}: {sorted(top_timesteps)}")

importance_per_channel = np.abs(sv_feat).sum(axis=0)
channel_names = ["acc_x","acc_y","acc_z","gyro_x","gyro_y","gyro_z","total_acc_x","total_acc_y","total_acc_z"]
plt.figure(figsize=(8, 3))
plt.bar(channel_names, importance_per_channel, color="steelblue")
plt.xlabel("Sensor Channel")
plt.ylabel("Sum |SHAP|")
plt.title(f"Channel Importance — {CLASS_NAMES[cls]}")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
import keras_tuner as kt
import tensorflow as tf
from tensorboard.plugins.hparams import api as hparams_api

def build_tunable(hp):
    conv1 = hp.Choice('conv1', [32, 64, 128])
    lstm_u = hp.Choice('lstm_u', [64, 128])
    dropout = hp.Float('drop', 0.1, 0.5, step=0.1)
    lr = hp.Choice('lr', [1e-3, 1e-4])

    model = build_cnn_lstm(
        conv_filters=[conv1, conv1*2],
        lstm_units=lstm_u,
        dropout=dropout
    )
    model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
    )
    return model

tuner = kt.RandomSearch(
    build_tunable,
    objective='val_accuracy',
    max_trials=5,
    overwrite=True,
    directory='tuner_results',
    project_name='har_tuning'
)

tuner.search(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=64,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
)

best_hp = tuner.get_best_hyperparameters()[0]
print("Best hyperparameters:")
print(f"  conv filters : {best_hp.get('conv1')} → {best_hp.get('conv1')*2}")
print(f"  lstm units   : {best_hp.get('lstm_u')}")
print(f"  dropout      : {best_hp.get('drop')}")
print(f"  learning rate: {best_hp.get('lr')}")

best_model = tuner.get_best_models(num_models=1)[0]
loss, acc = best_model.evaluate(X_test, y_test, verbose=0)
print(f"\nBest model test accuracy: {acc:.1%}")

baseline_loss, baseline_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Baseline accuracy : {baseline_acc:.1%}")
print(f"Tuned accuracy    : {acc:.1%}")
print(f"Improvement       : {(acc - baseline_acc)*100:+.2f}%")